# 01 — Baseline: XGBoost
Derin öğrenme modellerinin karşılaştırılacağı referans. Test = resmi `hidden_test` (10 ayrı kuyu).

In [ ]:
# --- Kurulum: Colab'da repoyu klonlar/günceller, yerelde repo köküne geçer ---
import os, sys, pathlib
REPO_URL = "https://github.com/aysenursimsk/force2020-litoloji.git"   # ← kendi repo adresiniz

if "google.colab" in sys.modules:
    if not pathlib.Path("/content/force2020-litoloji").exists():
        !git clone -q {REPO_URL} /content/force2020-litoloji
    %cd /content/force2020-litoloji
    !git pull -q

root = pathlib.Path.cwd()
if not (root / "src").exists():
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    print("autoreload kullanılamıyor (sorun değil): src değişirse oturumu yeniden başlatın.")

In [ ]:
KIM = ""          # ← adınız: deney kaydında görünür
N_WELLS = None    # hızlı deneme için ör. 20 (sadece ilk 20 eğitim kuyusu); final için None

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from xgboost import XGBClassifier
from src import veri, degerlendir as D

d = veri.hazirla(n_wells=N_WELLS)
d.ozet()

In [ ]:
ayar = dict(n_estimators=300, max_depth=8, learning_rate=0.1)
xgb = XGBClassifier(**ayar, tree_method="hist", n_jobs=-1, random_state=42)
xgb.fit(d.X_tr, d.y_tr, eval_set=[(d.X_va, d.y_va)], verbose=50)

p = xgb.predict(d.X_te)
s = D.skorlar(d.y_te, p); print(s)
D.rapor(d.y_te, p)
D.confusion(d.y_te, p, "XGBoost")

In [ ]:
pd.Series(xgb.feature_importances_, index=d.columns).sort_values().tail(15) \
  .plot.barh(figsize=(6, 5), title="XGBoost özellik önemi"); plt.show()

In [ ]:
D.kaydet("baseline", "XGBoost", ayar, s, kim=KIM, not_="ilk baseline")